---
# Extending Vector Database to More Emails
---

**Goal:** Expand the vector database with a subset of ENRON emails to enable semantic search and assess retrieval quality.

**Dataset:**

- Limited to 30,000 random emails from the ENRON dataset.

- Limiting the dataset helps manage computational resources and embedding creation time, even on GPU.

*Note:* Because this is a subset, search results may not always retrieve the most relevant emails from the full ENRON dataset, as some relevant emails may not be included in the sample.

**Embedding Considerations:**

Creating embeddings for all ENRON emails (~500k+) is time intensive, the subset of 30k is a compromise.

Added test emails to the dataset to verify that the vector search works as expected.


## Set Up
----

In [ ]:

import numpy as np
import joblib
import pandas as pd
import spacy
import re
from sentence_transformers import SentenceTransformer
from google.colab import files

## Functions
----

In [ ]:
def chunk_email(email, chunk_size=100, overlap=50):
    """Splits an email into chunks of a given size with a given overlap."""
    chunks = []
    words = email.split()
    step = chunk_size - overlap
    # Split email into chunks
    for i in range(0, len(words), step):
        chunk = words[i:i+chunk_size]
        chunked_email = ' '.join(chunk)
        chunks.append(chunked_email)

    return chunks

## Get Data
---

In [ ]:
uploaded = files.upload()

Saving cleaned_emails_EXT_TEST_3.csv to cleaned_emails_EXT_TEST_3.csv


In [ ]:
filename = list(uploaded.keys())[0]
cleaned_emails_df = pd.read_csv(filename, index_col = 0)

In [ ]:
cleaned_emails_df.shape

(24926, 7)

In [ ]:
cleaned_emails_df


,to,subject,body,cleaned_body,cleaned_subject,all_text,indecent_emails
from,,,,,,,
master.amar@hoegh.no,"dan.masters@enron.com, tony.galt@enron.com, ji...",POSREP 24 OCT,HOEGH GALLEON\t24-Oct 12:00LT\t24-Oct 10:00\t(...,HOEGH GALLEON 24Oct 1200LT 24Oct 1000 UTC A Po...,POSREP 24 OCT,POSREP 24 OCT HOEGH GALLEON 24Oct 1200LT at ...,False
marie.heard@enron.com,yolanda.cordova-gilbert@enron.com,GAF Refining LLC,Yolanda:\n\nIn connection with the execution o...,Yolanda In connection with the execution of th...,GAF Refining LLC,Refining LLC Yolanda In connection with the e...,False
darrell.schoolcraft@enron.com,"controllers.dl-ets@enron.com, kimberly.watson@...",FW: Gallup Compressor,I will let you know any updates. We will be a...,I will let you know any updates We will be all...,FW Gallup Compressor,I will let you know any updates We will be al...,False
frank.chmiel@us.abb.com,kay.mann@enron.com,Change Order #1 to ABB Purchase Agreement LM6K...,"Kay,\nThis note is to advise that I signed the...",Kay This note is to advise that I signed the r...,Change Order 1 to ABB Purchase Agreement LM6K2001,to Purchase Agreement LM6K2001 Kay This note...,False
kay.mann@enron.com,travis.mccullough@enron.com,Re: DASH today?,"Sorry, but I don't know. Probably in the next...",Sorry but I dont know Probably in the next 3 h...,Re DASH today,Re DASH today Sorry but I dont know Probably i...,False
...,...,...,...,...,...,...,...
lorraine.lindberg@enron.com,michelle.lokay@enron.com,Capacity,"Michelle - When you get a minute, let's go ove...",Michelle When you get a minute lets go over th...,Capacity,Capacity Michelle When you get a minute lets g...,False
justin.boyd@enron.com,andy.zipper@enron.com,RE: EnEx - New Online Gas Trading Platform,andy\n\nthanks for the note - i'll pass this o...,andy,RE EnEx New Online Gas Trading Platform,RE EnEx New Online Gas Trading Platform andy,False
iam@michaelberry.com,iam@michaelberry.com,MICHAEL BERRY campaign: 1) election night; 2) ...,Election Night Returns Party: on election nigh...,Election Night Returns Party on election night...,MICHAEL BERRY campaign 1 election night 2 earl...,campaign 1 election night 2 early vote party ...,False


In [ ]:
cleaned_emails_df.dropna(subset=['all_text'], inplace=True)

### Get Chunks

In [ ]:
cleaned_emails_df['all_text'] = cleaned_emails_df['all_text'].fillna("").astype(str)

In [ ]:
cleaned_emails_df.reset_index(drop=True, inplace=True)

In [ ]:
# to store text of chunked emails
chunked_emails = []
# to store index of the original email for each chunk
original_email_index = []

for index, email in enumerate(cleaned_emails_df['all_text']):
    chunks = chunk_email(email)
    chunked_emails.extend(chunks)

    original_email_index.extend([index] * len(chunks))

## Create Embeddings
---

In [ ]:
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
embeddings = model.encode(
    chunked_emails,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/1773 [00:00<?, ?it/s]

In [ ]:
embeddings.shape

(56718, 768)

## Save Data
----



In [ ]:
joblib.dump(original_email_index, 'EXT_original_email_index.pkl')

['EXT_original_email_index.pkl']

In [ ]:
joblib.dump(embeddings, "EXT_embeddings_chunked.pkl")

['EXT_embeddings_chunked.pkl']

In [ ]:

files.download("EXT_embeddings_chunked.pkl")
files.download("EXT_original_email_index.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cleaned_emails_df.head()

,to,subject,body,cleaned_body,cleaned_subject,all_text,indecent_emails
0,"dan.masters@enron.com, tony.galt@enron.com, ji...",POSREP 24 OCT,HOEGH GALLEON\t24-Oct 12:00LT\t24-Oct 10:00\t(...,HOEGH GALLEON 24Oct 1200LT 24Oct 1000 UTC A Po...,POSREP 24 OCT,POSREP 24 OCT HOEGH GALLEON 24Oct 1200LT at ...,False
1,yolanda.cordova-gilbert@enron.com,GAF Refining LLC,Yolanda:\n\nIn connection with the execution o...,Yolanda In connection with the execution of th...,GAF Refining LLC,Refining LLC Yolanda In connection with the e...,False
2,"controllers.dl-ets@enron.com, kimberly.watson@...",FW: Gallup Compressor,I will let you know any updates. We will be a...,I will let you know any updates We will be all...,FW Gallup Compressor,I will let you know any updates We will be al...,False
3,kay.mann@enron.com,Change Order #1 to ABB Purchase Agreement LM6K...,"Kay,\nThis note is to advise that I signed the...",Kay This note is to advise that I signed the r...,Change Order 1 to ABB Purchase Agreement LM6K2001,to Purchase Agreement LM6K2001 Kay This note...,False
4,travis.mccullough@enron.com,Re: DASH today?,"Sorry, but I don't know. Probably in the next...",Sorry but I dont know Probably in the next 3 h...,Re DASH today,Re DASH today Sorry but I dont know Probably i...,False


In [ ]:
cleaned_emails_df.to_csv('cleaned_emails.csv', index=False)


In [ ]:
files.download("cleaned_emails.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>